# 비만클리닉 doctor agent 만들기
**대한의료정보학회 2026 튜토리얼 실습** · 모든 환자 데이터는 합성 데이터입니다.

오늘은 **2026년 11월 25일**, 비만클리닉 진료일입니다. 세 환자가 옵니다.

| 환자 | 오늘 할 일 | 에이전트가 쓸 것 |
|---|---|---|
| **P1** (남, 27세) | 삭센다 7개월째 재진 | **재진 전 요약**: 진료실 들어가기 전 30초 안에 읽을 요약 |
| **P2** (여, 40세) | 처음 온 환자 | **초진기록 초안**: 의사 메모 + 문진표 + 기존 기록으로 |
| **P3** (남, 35세) | 위고비를 중단한 환자의 마지막 진료 | **치료 종료 기록**: 치료 전체를 지침 기준으로 정리 |

세 장면 모두 **같은 에이전트, 같은 도구**를 쓰고 **양식만 바꿉니다**. 오늘 만들 에이전트의 전체 모습입니다.

![에이전트 구조](https://raw.githubusercontent.com/chopresent/kosmi-tutorial-doctor-agent/main/figures/agent_structure.png)

코드는 모두 채워져 있습니다. 위에서부터 한 셀씩 실행하면서 결과와 주석(`#`)을 읽어 가면 됩니다.
🔁 표시가 있는 셀은 고쳐서 다시 돌려 보면 좋은 곳입니다(선택).

**결과 상자의 색**은 누가 만든 결과인지를 나타냅니다.

| 색 | 뜻 |
|---|---|
| 회색 🧑‍💻 | 코드가 실행한 결과. LLM을 부르지 않음 |
| 파란색 🤖 | LLM의 응답 (도구 요청서 또는 답) |
| 주황색 🔧 | LLM이 요청한 도구를 코드가 실행한 결과 |
| 초록색 📄 | 에이전트의 최종 산출물 |

| 순서 | 내용 |
|---|---|
| 0 | 준비: 자료 내려받기, API 키, 글씨 크기 |
| 1 | LLM 한 번 불러 보기 |
| 2 | 데이터 보기: 왜 그대로 넣지 않는가 |
| 3 | 도구: 읽기와 계산은 코드가 한다 (LLM 없음) |
| 4 | LLM이 도구를 쓰는 방법: 요청서 한 장 |
| 5 | 에이전트 루프: 반복문 하나 (P1 재진 요약) |
| 6 | 결과 점검하기, 프롬프트 한 줄의 힘 |
| 7 | 초진기록: 양식만 바꾸기 (P2) |
| 8 | 치료 종료 기록 (P3) |
| 9 | 시간이 남으면 (선택 블록 7개) |
| 정리 | 원내로 옮긴다면, 토론 |

## 0. 준비

**① 이 노트북을 내 Drive에 사본으로 저장하세요** (메뉴 `파일 → Drive에 사본 저장`). 그래야 고친 내용이 남습니다.

**② API 키 등록**: 왼쪽 막대의 🔑(보안 비밀) → `새 보안 비밀 추가` → 이름 `GEMINI_API_KEY`, 값에 키 붙여넣기 → `노트북 액세스` 켜기.
키는 [Google AI Studio](https://aistudio.google.com/apikey)에서 무료로 만듭니다(결제 정보 필요 없음). **키를 코드 셀에 직접 붙여넣지 마세요.** 노트북을 공유할 때 같이 나갑니다.

**③ 글씨가 작으면**
- 화면 전체: 브라우저 확대 `Ctrl` + `+` (Mac은 `⌘` + `+`). 125~150%를 권합니다. 되돌리기는 `Ctrl` + `0`
- 코드 글씨만: 메뉴 `도구 → 설정 → 편집기 → 글꼴 크기`
- 결과 상자 글씨: 1번 셀 뒤에 `ka.FONT_PX = 18` 처럼 숫자를 바꾸면 커집니다(기본 16)

첫 셀을 실행하면 "Google에서 작성하지 않은 노트북" 경고가 뜹니다. `무시하고 계속`을 누르세요.

In [ ]:
# 실습 자료(코드·합성 데이터)를 GitHub에서 내려받아 /content/kosmi 에 풉니다. 30초 정도 걸립니다.
import os, sys, zipfile, urllib.request

REPO_ZIP = "https://github.com/chopresent/kosmi-tutorial-doctor-agent/archive/refs/heads/main.zip"
PKG = os.environ.get("KOSMI_PKG_DIR", "/content/kosmi")           # 자료를 풀 폴더

if not os.path.exists(os.path.join(PKG, "kosmi_tools.py")):         # 이미 받았으면 건너뜀
    urllib.request.urlretrieve(REPO_ZIP, "/content/kosmi.zip")
    with zipfile.ZipFile("/content/kosmi.zip") as z:
        z.extractall("/content")
        top = z.namelist()[0].split("/")[0]                          # 압축 안의 최상위 폴더 이름
    os.rename(os.path.join("/content", top), PKG)
sys.path.insert(0, PKG)                                              # 실습 모듈을 import 할 수 있게

!pip -q install openai > /dev/null
print("준비 완료:", sorted(f for f in os.listdir(PKG) if not f.startswith(".")))

In [ ]:
# 🔑 보안 비밀에 넣어 둔 API 키를 환경 변수로 불러옵니다.
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("🔑 키를 불러왔습니다.")
except ImportError:                     # Colab이 아닌 곳에서 실행할 때
    if not os.environ.get("GEMINI_API_KEY"):
        import getpass
        os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")
except Exception as e:
    print("⚠️ 키를 찾지 못했습니다. 왼쪽 🔑에 GEMINI_API_KEY를 넣고 '노트북 액세스'를 켰는지 확인하세요.", e)

## 1. LLM 한 번 불러 보기

Gemini를 **OpenAI 호환 방식**으로 부릅니다. 원내 서버(vLLM 등)로 옮길 때는 아래 `base_url`, `api_key`, 모델 이름 **세 가지만** 바뀝니다.
무료 등급 한도는 모델별로 **분당 15회, 하루 500회**입니다. 이 노트북 전체를 돌려도 15회 안팎입니다.

In [ ]:
from openai import OpenAI
import kosmi_tools as kt      # 데이터, 도구
import kosmi_agent as ka      # 모델 호출, 프롬프트·양식, 자동 점검, 결과 상자(ka.show)

client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",   # 원내 vLLM이면 "http://내부서버:8000/v1"
    api_key=os.environ.get("GEMINI_API_KEY") or "키-없음",                 # 키가 없어도 아래 '녹화 재생'으로 따라갈 수 있음
)
print("사용 모델:", ka.MODELS)   # 첫 모델의 하루 한도가 끝나거나 응답이 45초를 넘기면 다음 모델로 자동 전환

**API가 안 될 때** (키 발급 실패, 서버 장애, 네트워크 문제): 아래 셀의 맨 앞 `#`을 지우고 실행하세요.
강의 전에 실제 Gemini로 녹화해 둔 응답을 대신 써서 노트북 끝까지 따라갈 수 있습니다. 도구 실행과 점검은 그대로 실제로 돕니다.
- 녹화는 **원래 코드·질문 그대로일 때만** 맞습니다. 🔁 셀을 고치면 재생되지 않습니다.
- 이 셀을 건너뛰어도, API 호출이 실패하면 녹화가 있는 셀은 자동으로 녹화로 대신합니다(▶️ 표시).

In [ ]:
# ka.REPLAY = True          # ← 맨 앞 #을 지우면 녹화 재생 (API를 부르지 않음)
print("녹화 재생:", "켜짐 ▶️" if ka.REPLAY else "꺼짐 (실제 API 사용)")

In [ ]:
# 🔁 질문을 바꿔서 다시 실행해 보세요.
# ka.chat()은 client.chat.completions.create()를 감싼 함수입니다.
# 분당 한도(429)에 걸리면 잠시 쉬었다가 다시 부르고, 하루 한도가 끝나면 다른 무료 모델로 넘어갑니다.
r = ka.chat(client, [{"role": "user", "content": "GLP-1 유사체 비만약의 흔한 부작용을 세 가지만 한 줄씩 알려줘"}])
ka.show("llm", r.choices[0].message.content, title="LLM의 답")

여기까지는 우리가 아는 채팅과 같습니다. 그런데 "이 환자 체중이 얼마나 줄었어?"라고 물으면 LLM은 답할 수 없습니다. **이 환자의 기록을 본 적이 없기 때문입니다.** 지금부터 이 간격을 메웁니다.

## 2. 데이터 보기: 왜 그대로 넣지 않는가

세 환자의 기록이 표 7개로 들어 있습니다.

| 표 | 내용 |
|---|---|
| `patients` | 성별, 출생연도 |
| `diagnoses` | 진단 (KCD 코드) |
| `prescriptions` | 비만약 처방: 약, 용량, 처방일수 |
| `vitals` | 체중·키·BMI·혈압 (측정 1회 = 1행) |
| `questionnaire` | 환자가 작성한 체중관리 문진표 21문항 |
| `notes` | 이전 진료기록(초진기록)의 자유서술 |
| `labs` | 혈액검사 결과 |

In [ ]:
db = kt.load_data()                              # 표 7개를 dict로 읽음: db["vitals"] 등
for name, df in db.items():
    print(f"{name:14s} {len(df):4d}행   열: {list(df.columns)}")

In [ ]:
# 오늘 첫 환자 P1의 기록을 표마다 모아 봅니다.
rows = {name: df[df["patient_id"] == "P1"] for name, df in db.items()}
print({name: len(r) for name, r in rows.items()})

rows["vitals"].tail(8)          # 활력징후 마지막 8행: 체중은 측정한 날만, 혈압은 하루에 여러 번 잰 날도 있음

이 정도면 통째로 LLM에 넣어도 될 것 같지만, 그렇게 하지 않습니다.
1. **필요한 것만 골라야 한다**: 실습 데이터는 작지만, 실제 EMR은 수년간의 진료·검사·간호기록이 쌓여 금방 수십만 자가 됩니다.
2. **계산을 LLM이 하게 된다**: 체중 변화율, 증량 단계, 지침 판정을 LLM이 표를 보고 암산하면 틀릴 수 있습니다.

그래서 **읽기와 계산은 코드(도구)가 하고, LLM은 어떤 도구를 쓸지 고르고 결과를 글로 정리만** 하게 나눕니다.

## 3. 도구: 읽기와 계산은 코드가 한다 (LLM 없음)

**도구는 평범한 파이썬 함수입니다.** 기록 표를 읽고, 정해진 규칙대로 계산하고, 결과를 글로 돌려줍니다.
이 섹션의 셀은 **LLM을 한 번도 부르지 않습니다.** 같은 환자에게 몇 번을 돌려도 결과가 똑같습니다.

실습에는 도구 8개를 만들어 두었습니다.

| 도구 | 하는 일 |
|---|---|
| `get_patient_summary` | 성별·나이·키, 비만클리닉 등록일, 진단 목록, 방문 수 |
| `get_weight_trend` | 기준 체중, 최근 체중, **변화율(%)**, 최저 체중, BMI와 비만 단계 |
| `get_medication_timeline` | 비만약 처방 타임라인: 날짜별 용량, **처방 소진일**, 현재·다음 증량 단계 |
| `get_vitals` | 최근 혈압 목록과 140/90 이상 횟수 |
| `get_questionnaire` | 환자가 쓴 문진표: 질환·복용약·음주·수면, 졸림·우울 점수와 해석 |
| `get_previous_notes` | 이전 진료기록의 주호소·현병력·평가 및 계획 |
| `get_labs` | 혈액검사 결과(지질, 혈당, HbA1c, 간기능, TSH) |
| `assess_response` | 대한비만학회 2022 기반 실습 정의로 **약물 반응·6개월 목표 판정** |

도구를 쓰려면 먼저 환자를 엽니다. `kt.Context("P1")`은 **진료할 환자 한 명의 차트를 여는 것**과 같습니다.
도구는 이렇게 연 환자에게 **묶여** 있어서, 나중에 LLM이 무슨 말을 하든 다른 환자의 기록을 볼 방법이 없습니다.

In [ ]:
ctx = kt.Context("P1")              # 오늘 첫 환자 P1의 차트를 연다 (P1의 오늘 할 일 = 재진 요약)
print(ctx.header())                 # [환자 · 오늘 할 일 · 오늘 날짜]

이제 도구를 **우리가 직접** 불러 봅니다. 함수 호출일 뿐입니다.

In [ ]:
ka.show("code", kt.get_weight_trend(ctx), title="kt.get_weight_trend(ctx)")   # 체중 도구: 변화율·최저 체중을 코드가 계산

도구 결과에는 이미 **출처 태그**(`[기록 2026-11-14]`)가 붙어 있습니다. 나중에 LLM은 이 태그를 옮겨 적기만 하면 됩니다.

아래는 **판정 도구**입니다. "유지 용량 3개월 시점에 5% 이상 감량했는가"를 정의대로 계산합니다.
🔁 `get_medication_timeline`, `get_questionnaire`, `get_previous_notes`, `get_labs`로 바꿔서도 실행해 보세요.

In [ ]:
ka.show("code", kt.assess_response(ctx), title="kt.assess_response(ctx)")

## 4. LLM이 도구를 쓰는 방법: 요청서 한 장

LLM은 우리 컴퓨터에 있는 함수를 **직접 실행할 수 없습니다.** LLM이 할 수 있는 일은 글을 쓰는 것뿐입니다.
그래서 도구가 필요하면 "`get_weight_trend`를 실행해 주세요"라는 **요청서**(도구 이름 + 인자)를 글로 써서 돌려줍니다.
실행은 우리 코드가 하고, 그 결과를 다시 LLM에게 보냅니다. 한 번의 왕복은 아래 6단계입니다.

![요청서 왕복](https://raw.githubusercontent.com/chopresent/kosmi-tutorial-doctor-agent/main/figures/tool_call.png)

여기서 꼭 알아 둘 것이 하나 있습니다. **LLM은 앞서 나눈 대화를 기억하지 못합니다.**
그래서 우리 코드가 주고받은 메시지를 `conversation`이라는 **대화 기록 목록**에 모아 두고, LLM을 부를 때마다 이 목록을 통째로 보냅니다.
아래 셀들을 따라가면 대화 기록이 **1줄 → 3줄**로 늘어나는 것을 볼 수 있습니다.

**먼저 메뉴판.** LLM에게는 도구의 *이름과 설명*만 알려 줍니다. 함수 코드는 보내지 않습니다.
그래서 **설명(description)이 곧 사용 설명서**입니다.

In [ ]:
# 메뉴판 한 줄: 도구 이름, 설명, 받는 인자(JSON Schema). 환자는 이미 열려 있으니 인자가 없습니다.
weight_tool = {
    "type": "function",
    "function": {
        "name": "get_weight_trend",
        "description": "체중 추이: 기준 체중, 최근 체중, 변화율(%), 최저 체중, BMI. 체중·변화율은 반드시 이 도구 값만 쓴다.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    },
}

In [ ]:
# ❶ 대화 기록 1줄(질문)과 메뉴판(tools=)을 보낸다
conversation = [{"role": "user", "content": "이 환자의 체중이 등록 후 얼마나 줄었나요?"}]
ka.show_conversation(conversation, title="❶ LLM에게 보내는 대화 기록 (1줄) + 메뉴판 1줄")

r = ka.chat(client, conversation, tools=[weight_tool])
msg = r.choices[0].message                        # ❷ LLM의 응답
if not msg.tool_calls:                            # 드물게 요청서 없이 바로 답하기도 함
    raise RuntimeError(f"이번에는 요청서 없이 답했습니다: {msg.content!r}  → 이 셀을 한 번 더 실행해 보세요.")

request = msg.tool_calls[0]                       # 요청서 (여러 장일 수도 있음)
ka.show("llm", f"답: {msg.content or '(비어 있음)'}\n"
               f"요청서: 도구 이름 = {request.function.name}, 인자 = {request.function.arguments}",
        title="❷ LLM의 응답: 답 대신 요청서")

In [ ]:
# ❸ 우리 코드가 요청서에 적힌 도구를 실행한다  →  ❹ 결과
# kt.call_tool은 이름을 보고 해당 함수를 부르는 작은 함수입니다: get_weight_trend(ctx)
result = kt.call_tool(ctx, request.function.name, request.function.arguments)
ka.show("tool", result, title=f"❹ {request.function.name} 실행 결과")

### ❺ 대화 기록에 두 줄을 덧붙인다
이제 결과를 LLM에게 알려 줘야 합니다. 그런데 LLM은 ❷에서 **자기가 요청서를 썼다는 것조차 기억하지 못합니다.**
그래서 결과만 보내면 안 되고, 대화 기록에 두 줄을 덧붙여 **질문 → 요청서 → 결과** 3줄을 한꺼번에 보냅니다.

In [ ]:
# 대화 기록(conversation)은 지금 1줄(질문)입니다. 두 줄을 덧붙여 3줄로 만듭니다.
conversation.append(msg.model_dump(exclude_none=True))   # 2줄: LLM이 ❷에서 쓴 요청서 (목록에 넣기 좋은 형태로 바꿔서)
conversation.append({                                     # 3줄: ❹의 도구 결과
    "role": "tool",                                       #   이 줄은 '도구 결과'라는 표시
    "tool_call_id": request.id,                           #   어느 요청서에 대한 결과인지 (요청서 번호)
    "content": result,                                    #   결과 내용
})
ka.show_conversation(conversation, title="❺ LLM에게 다시 보낼 대화 기록 (3줄)")

### ❻ 3줄을 다시 보낸다
같은 질문이지만 이번에는 대화 기록 안에 도구 결과가 이미 있습니다. 그래서 LLM은 요청서 대신 **답**을 씁니다.

In [ ]:
r = ka.chat(client, conversation, tools=[weight_tool])     # ❶과 같은 호출, 대화 기록만 3줄로 늘었음
ka.show("llm", r.choices[0].message.content, title="❻ LLM의 답 (이번엔 요청서 없음)")

LLM은 도구를 **실행하지 않았습니다.** 요청서를 썼고, 실행과 결과 전달은 우리 코드가 했습니다.
LLM이 쓴 숫자(-4.0%)는 LLM이 계산한 것이 아니라 ❹의 도구 결과를 옮긴 것입니다.

이 왕복을 **LLM이 요청서 없이 답을 낼 때까지 자동으로 반복**하는 것이 다음 섹션의 에이전트 루프입니다.
실제 에이전트에게는 메뉴판에 도구 8개를 모두 줍니다.

In [ ]:
menu = [f"- {t['function']['name']}: {t['function']['description'][:60]}" for t in kt.TOOL_SPECS]
ka.show("code", "\n".join(menu), title="에이전트에게 줄 메뉴판 8줄 (kt.TOOL_SPECS)")

## 5. 에이전트 루프: 반복문 하나

![에이전트 루프](https://raw.githubusercontent.com/chopresent/kosmi-tutorial-doctor-agent/main/figures/agent_loop.png)

아래가 에이전트 전부입니다. 그림의 ①~⑤가 코드 주석의 ①~⑤와 같습니다.
실행하면 **파란 상자**(LLM이 무엇을 요청했나)와 **주황 상자**(코드가 도구를 실행한 결과)가 번갈아 나오고, 마지막에 **초록 상자**(최종 산출물)가 나옵니다.

In [ ]:
def my_agent(ctx, messages, tools=kt.TOOL_SPECS, max_steps=8):
    trace = []                                                  # 어떤 도구를 불렀는지 기록 (나중에 점검용)
    for step in range(1, max_steps + 1):                        # 무한 반복을 막기 위해 최대 8번
        r = ka.chat(client, messages, tools=tools)              # ① 대화 전체 + 메뉴판(도구 목록)을 LLM에 보냄
        msg = r.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))      # ② LLM의 응답을 대화에 그대로 덧붙임

        if not msg.tool_calls:                                  # ③ 요청서가 없으면 = 최종 답
            ka.show("llm", "요청서 없음 → 최종 답을 씀", title=f"{step}번째 LLM 호출")
            return msg.content, trace

        ka.show("llm", "요청서 " + str(len(msg.tool_calls)) + "장: " + ", ".join(tc.function.name for tc in msg.tool_calls),
                title=f"{step}번째 LLM 호출")
        done = []
        for tc in msg.tool_calls:                               # ④ 요청서마다 도구를 실행 (코드가 함, LLM 아님)
            result = kt.call_tool(ctx, tc.function.name, tc.function.arguments)   # 도구는 ctx(지금 연 환자)에 묶여 있음
            trace.append((step, tc.function.name, tc.function.arguments, result))
            done.append(f"{tc.function.name:24s} → {result.splitlines()[1][:48]}")
            messages.append({"role": "tool", "tool_call_id": tc.id,      # ⑤ 결과를 '어느 요청서의 답인지' 표시해 돌려줌
                             "content": result})
        ka.show("tool", "\n".join(done), title=f"도구 {len(done)}개 실행 (첫 줄만 표시)")
    return "(단계 초과)", trace

### 첫 실행: P1 재진 전 요약
`ka.build_messages()`는 **시스템 프롬프트**(규칙·지침 요점)와 **재진 요약 양식**, 요청 문장을 묶어 줍니다.

In [ ]:
print(ka.SYSTEM_COMMON)      # 시스템 프롬프트: 출처 태그 규칙, 계산값은 도구 값만, 지침 요점

In [ ]:
ctx = kt.Context("P1")
messages = ka.build_messages(ctx)          # 시스템 프롬프트 + 재진 요약 양식 + 요청
out, trace = my_agent(ctx, messages)       # 파란 상자 = LLM, 주황 상자 = 도구 실행
ka.show("final", out, title="P1 재진 전 요약")
p1_messages, p1_out = messages, out      # 9번 선택 블록에서 다시 씀

**살펴볼 것**
- LLM을 몇 번 불렀나요? 보통 첫 호출에서 요청서를 여러 장 한꺼번에 쓰고, 두 번째 호출에서 글을 씁니다.
- 어떤 도구를 부를지는 **코드에 적지 않았습니다.** 양식을 보고 LLM이 골랐습니다.
- 초록 상자의 숫자 하나를 골라, 아래 주황 상자(도구 결과 전체)에서 같은 숫자와 날짜를 찾아보세요.

In [ ]:
ka.show_trace(trace)            # 도구 결과 앞부분. 전체를 보려면 ka.show_trace(trace, full=True)

## 6. 결과 점검하기, 프롬프트 한 줄의 힘

기록 문서는 정답이 하나가 아닙니다. 대신 **사실 추적성**은 자동으로 점검할 수 있습니다.
- **출처 표시**: 숫자가 있는 줄마다 출처 태그가 있나
- **날짜 근거**: 오늘 이후의 날짜를 지어내지 않았나 (도구가 계산한 처방 소진일 등은 통과)
- **수치 근거**: 쓴 숫자가 도구 결과·입력에 실제로 있나

In [ ]:
ka.print_check(ka.check_output(ctx, out, trace))       # 코드가 검사 (LLM 없음)

P1은 BMI 41의 3단계 비만인데 약물치료에 반응이 없었습니다. 시스템 프롬프트의 **[지침 요점]**에는 비만대사수술 상담 기준이 한 줄 들어 있습니다.
그 줄을 빼고 돌리면 "오늘 확인할 것"이 어떻게 달라지는지 보세요.

In [ ]:
# 시스템 프롬프트에서 수술 상담 기준 한 줄만 지운 버전
surgery_line = "- BMI 35 이상, 또는 30 이상이면서 동반질환이 있는데 생활습관·약물치료로 감량이 부족하면 비만대사수술 상담을 고려한다.\n"
no_surgery = ka.SYSTEM_COMMON.replace(surgery_line, "")

out2, trace2 = my_agent(ctx, ka.build_messages(ctx, system=no_surgery))
ka.show("final", out[out.find("오늘 확인할 것"):], title="지침 줄 있음")
ka.show("final", out2[out2.find("오늘 확인할 것"):], title="지침 줄 없음")

모델이 의학 지식이 없어서가 아니라, **우리 기관이 무엇을 확인하길 원하는지**를 프롬프트로 알려 주지 않았기 때문입니다.

## 7. 초진기록: 양식만 바꾸기 (P2)

두 번째 환자 P2(40세 여성)는 처음 왔습니다. 진료를 마친 의사가 남긴 짧은 메모입니다.

In [ ]:
memo = ka.load_note("P2")                # 진료 중 의사가 남긴 메모
print(memo)

같은 에이전트·같은 도구에 **양식**과 **의사 메모**만 바꿔 넣습니다.

이 메모는 문진표·기록과 **세 군데 어긋나도록** 만들어 두었습니다.
- 기저질환: 메모 "없음" ↔ 문진표 "빈혈"
- 음주: 메모 "회식 때 가끔" ↔ 문진표 "마시지 않음"
- 체중: 메모 "1년 4kg↑" ↔ 기록상 1년간 +0.9kg

초진기록 양식은 두 가지를 준비했습니다. 차이는 맨 끝의 **'메모·문진표 대조' 칸** 하나뿐입니다.

In [ ]:
print(ka.CHECK_BLOCK)                    # FORM_INITIAL = FORM_INITIAL_NOCHECK + 이 칸

In [ ]:
import re

def found_mismatch(out):
    """세 가지 불일치를 '확인 필요'로 표시했는지 줄 단위로 확인 (코드가 셈)"""
    keys = {"기저질환(빈혈)": "빈혈|기저질환", "음주": "음주", "체중 변화": r"체중 변화|4\s*kg"}
    return {name: any(re.search(pat, line) and re.search("확인|불일치", line) for line in out.splitlines())
            for name, pat in keys.items()}

ctx = kt.Context("P2")                   # 두 번째 환자 P2를 연다 (오늘 할 일 = 초진기록)

# ① 대조 칸이 없는 양식
print("━━━━━━━━ ① 대조 칸이 없는 양식으로 실행 ━━━━━━━━")
out_plain, _ = my_agent(ctx, ka.build_messages(ctx, memo, form=ka.FORM_INITIAL_NOCHECK))
# ② 대조 칸이 있는 양식 (form을 안 주면 ka.FORM_INITIAL)
print("━━━━━━━━ ② 대조 칸이 있는 양식으로 실행 ━━━━━━━━")
out_checked, trace = my_agent(ctx, ka.build_messages(ctx, memo))

mark = lambda d: "   ".join(f"{k} {'✅' if v else '❌'}" for k, v in d.items())
ka.show("code", f"대조 칸 없음: {mark(found_mismatch(out_plain))}\n대조 칸 있음: {mark(found_mismatch(out_checked))}",
        title="불일치 3곳을 찾았나")

In [ ]:
ka.show("final", out_checked, title="P2 초진기록 초안 (대조 칸 있음)")
ka.print_check(ka.check_output(ctx, out_checked, trace, memo))

"어긋나면 확인 필요라고 써라"라는 규칙은 시스템 프롬프트에 이미 있습니다. 그래도 대조 칸이 없으면 자주 놓칩니다(리허설: 평균 3곳 중 1곳).
**비교할 짝을 출력 칸으로 만들어 하나씩 쓰게 하면** 빠뜨리지 않습니다(리허설: 매번 3곳). LLM이 자주 놓치는 판단은 "잘 해라"가 아니라 **절차를 양식에 넣는 것**으로 안정시킵니다.

🔁 **선택**: 메모를 고쳐서(예: `memo = memo.replace("술은 회식 때 가끔", "술 안 마심")`) 다시 돌려 보세요. 대조 결과가 어떻게 바뀌나요?

## 8. 치료 종료 기록 (P3)

세 번째 환자 P3는 위고비를 쓰다가 그만둔 환자입니다. 오늘이 마지막 진료입니다.
P3는 체중이 **5.7% 줄었지만** 위고비 1.0mg(유지 용량 1.7mg 전)에서 중단했습니다. 실습 정의상 약물 반응은 **'판정 불가'**입니다. 에이전트가 숫자에 끌려 '반응'이라고 쓰지 않는지 보세요.

In [ ]:
ctx = kt.Context("P3")                   # 세 번째 환자 P3을 연다 (오늘 할 일 = 치료 종료 기록)
out, trace = my_agent(ctx, ka.build_messages(ctx))
ka.show("final", out, title="P3 치료 종료 기록")
ka.print_check(ka.check_output(ctx, out, trace))

판정은 LLM이 아니라 `assess_response` 도구가 정의대로 계산합니다. LLM의 역할은 그 결과를 **바꾸지 않고 옮기는 것**입니다.

In [ ]:
ka.show("code", kt.assess_response(ctx), title="판정의 원본: kt.assess_response(ctx)")

## 9. 시간이 남으면 (선택 블록)

블록마다 3~5분입니다. 서로 독립적이라 **아무 블록이나 골라서** 실행해도 됩니다(섹션 5까지는 실행돼 있어야 함).
시간이 없으면 건너뛰고 맨 아래 **정리**로 가세요.

| 블록 | 내용 | 이어지는 섹션 |
|---|---|---|
| 9-1 | 도구 없이 표를 통째로 주면? | 2 |
| 9-2 | 에이전트가 실제로 본 대화 기록 전체 | 4·5 |
| 9-3 | 우리 기관 규칙 한 줄 더하기 | 6 |
| 9-4 | 판정 도구를 메뉴판에서 빼면? | 8 |
| 9-5 | 같은 에이전트, 다른 독자: 환자용 안내문 | 7 (양식) |
| 9-6 | 모델만 바꿔 보기 | 1 (원내 이식) |
| 9-7 | 새 도구 만들기 | 3·4 |

### 9-1. 도구 없이 표를 통째로 주면?
섹션 2에서 "기록을 통째로 넣지 않는다"고 했습니다. 실제로 넣어 보겠습니다. P1의 활력징후 표(32행, 빈칸 포함)를 글자 그대로 주고 변화율을 묻습니다. **도구도 메뉴판도 없습니다.**

In [ ]:
ctx = kt.Context("P1")
raw = ctx.t["vitals"][["date", "weight_kg", "sbp", "dbp"]].to_csv(index=False)    # 표를 CSV 글자 그대로
question = (f"아래는 한 환자의 활력징후 기록(CSV)입니다. 등록일은 {ctx.registration_date():%Y-%m-%d}입니다.\n"
            "등록일 체중 대비 가장 최근 체중의 변화율(%)과 최저 체중(kg)을 숫자로만 알려 주세요.\n\n" + raw)
r = ka.chat(client, [{"role": "user", "content": question}])                      # tools 없음
ka.show("llm", r.choices[0].message.content, title="도구 없이: 표를 통째로 받은 LLM의 답")
ka.show("code", kt.get_weight_trend(ctx), title="도구가 계산한 값 (같은 표에서 코드가 계산)")

맞았나요, 틀렸나요? 맞았더라도 **맞았는지 확인하려면 결국 코드가 계산한 값이 필요합니다.** 옆 사람과 답이 같은지도 비교해 보세요. 그리고 실제 EMR이라면 이 표가 수년 치, 수십 개입니다.

### 9-2. 에이전트가 실제로 본 대화 기록 전체
섹션 4에서 대화 기록이 1줄 → 3줄로 늘어나는 것을 봤습니다. 섹션 5의 P1 에이전트가 마지막에 LLM에게 보낸 대화 기록을 열어 봅니다.

In [ ]:
ka.show_conversation(p1_messages, title="P1 재진 요약: 에이전트가 주고받은 대화 기록 전체")

시스템 프롬프트 1줄, 요청 1줄, 요청서 1줄, 도구 결과 8줄, 최종 답 1줄입니다. LLM이 본 것은 **이것이 전부**입니다. 기록에 없는 사실을 쓰면, 그건 LLM이 지어낸 것입니다.

### 9-3. 우리 기관 규칙 한 줄 더하기
섹션 6에서는 지침 한 줄을 **뺐습니다.** 이번에는 우리 클리닉만의 규칙을 한 줄 **더합니다.** P1은 혈액검사 기록이 하나도 없습니다.

In [ ]:
lab_rule = "- 비만약을 쓰는 중인데 최근 6개월 안에 혈액검사(지질·공복혈당·HbA1c·간기능) 결과가 없으면 검사를 계획에 넣는다.\n"
mood_line = "- 불안·우울 척도가 중등도(11점) 이상이면 기분 평가를 계획에 넣는다.\n"
my_system = ka.SYSTEM_COMMON.replace(mood_line, mood_line + lab_rule)     # 지침 요점 맨 끝에 한 줄 추가

ctx = kt.Context("P1")
out3, _ = my_agent(ctx, ka.build_messages(ctx, system=my_system))
ka.show("final", p1_out[p1_out.find("오늘 확인할 것"):], title="원래 프롬프트")
ka.show("final", out3[out3.find("오늘 확인할 것"):], title="우리 규칙 한 줄 추가")

### 9-4. 판정 도구를 메뉴판에서 빼면?
P3는 -5.7%인데 유지 용량 전에 중단해서 '판정 불가'였습니다. 그 판정은 `assess_response` 도구가 했습니다. 이 도구를 메뉴판에서 빼면 LLM은 무엇이라고 쓸까요?

In [ ]:
import re
no_judge = [t for t in kt.TOOL_SPECS if t["function"]["name"] != "assess_response"]    # 메뉴판 7줄

ctx = kt.Context("P3")
out4, _ = my_agent(ctx, ka.build_messages(ctx), tools=no_judge)
judge_lines = [l for l in out4.splitlines() if re.search("반응|목표|판정", l)]
ka.show("final", "\n".join(judge_lines), title="판정 도구 없이 쓴 판정 관련 줄")
ka.show("code", kt.assess_response(ctx), title="판정 도구가 계산한 원본")

'판정 불가'를 지켰나요, 아니면 -5.7%를 보고 '반응'이라고 썼나요? 판정 기준을 아는 것과 **정의대로 계산하는 것**은 다릅니다. 지켰더라도 근거(유지 용량 날짜, 84±28일 창)를 정확히 쓰기는 어렵습니다.

### 9-5. 같은 에이전트, 다른 독자: 환자용 안내문
양식만 바꾸면 같은 에이전트가 환자에게 드릴 안내문도 씁니다. 숫자와 판정은 도구 그대로, 말투만 바뀝니다.

In [ ]:
FORM_PATIENT = """[양식: 환자에게 드리는 안내문 — 쉬운 말, 6줄 안팎]
<의학 용어는 쉬운 말로 풀어 쓴다. 숫자와 출처 태그는 그대로 둔다. 약을 바꾸라거나 끊으라고 직접 권하지 않는다>
[{환자} 환자분께 · {오늘}]

지금까지의 변화: <체중 변화를 한 문장으로>
약의 효과: <assess_response 판정을 쉬운 말로>
오늘 선생님과 이야기할 것: <1~3개>"""

ctx = kt.Context("P1")
out5, _ = my_agent(ctx, ka.build_messages(ctx, form=FORM_PATIENT))
ka.show("final", out5, title="P1 환자용 안내문")

### 9-6. 모델만 바꿔 보기
원내 서버로 옮기면 모델이 바뀝니다. 코드는 그대로 두고 모델 이름만 바꿔 P1을 다시 돌려 봅니다(여기서는 무료 예비 모델 3.1 Flash-Lite).
판정은 도구가 하니 같아야 하고, 문장과 출처 태그는 달라질 수 있습니다.

In [ ]:
saved = ka.MODELS
ka.MODELS = ["gemini-3.1-flash-lite"]            # 원내라면 "내부-모델-이름"
try:
    ctx = kt.Context("P1")
    out6, trace6 = my_agent(ctx, ka.build_messages(ctx))
finally:
    ka.MODELS = saved                            # 원래 모델로 되돌림
ka.show("final", out6, title="P1 재진 요약 — 3.1 Flash-Lite")
ka.print_check(ka.check_output(ctx, out6, trace6))

### 9-7. 새 도구 만들기
도구는 평범한 함수 + 메뉴판 한 줄입니다. "6개월 목표 체중까지 얼마나 남았나"를 계산하는 도구를 만들어 메뉴판에 추가합니다.

In [ ]:
def get_weight_goal(ctx):
    """6개월 목표: 기준 체중의 5%·10% 감량 체중과 남은 kg (코드가 계산)"""
    w = ctx._weights()                                      # 날짜별 체중 표 (date, kg)
    base_date, base_kg = kt._nearest(w, ctx.registration_date(), 14)   # 등록일 ±14일의 기준 체중
    last = w.iloc[-1]                                       # 가장 최근 체중
    lines = [ctx.header(),
             f"기준 체중 {base_kg:.1f}kg [기록 {base_date:%Y-%m-%d}]",
             f"최근 체중 {last['kg']:.1f}kg [기록 {last['date']:%Y-%m-%d}]"]
    for pct in (5, 10):
        goal = base_kg * (1 - pct / 100)
        lines.append(f"{pct}% 감량 목표 {goal:.1f}kg → 남은 {max(0, last['kg'] - goal):.1f}kg")
    return "\n".join(lines)

goal_spec = {"type": "function", "function": {
    "name": "get_weight_goal",
    "description": "6개월 목표 체중(기준 체중의 5%·10% 감량)과 목표까지 남은 kg. 목표 관련 숫자는 이 도구 값만 쓴다.",
    "parameters": {"type": "object", "properties": {}, "required": []}}}

kt.TOOL_FUNCS["get_weight_goal"] = get_weight_goal       # ① 함수 등록 (call_tool이 이름으로 찾음)
my_tools = kt.TOOL_SPECS + [goal_spec]                   # ② 메뉴판에 한 줄 추가

ctx = kt.Context("P1")
ka.show("code", get_weight_goal(ctx), title="새 도구를 직접 실행 (LLM 없음)")
out7, _ = my_agent(ctx, [{"role": "user", "content": "이 환자가 6개월 목표(5%, 10% 감량)까지 각각 몇 kg 남았는지 두 문장으로 알려 주세요. 숫자는 도구 결과만 쓰세요."}],
                   tools=my_tools)
ka.show("final", out7, title="새 도구를 쓴 에이전트의 답")

LLM이 메뉴판에서 새 도구를 골라 썼나요? 원내에서 도구를 늘리는 방법도 이와 같습니다. **함수 하나 + 메뉴판 한 줄.**

In [ ]:
print(ka.usage_summary())         # 이 노트북에서 쓴 LLM 호출 수와 토큰

## 정리: 원내로 옮긴다면

| 오늘 실습 | 원내 배포 |
|---|---|
| Gemini 무료 API (합성 데이터) | 원내 GPU 서버의 오픈 웨이트 모델 (vLLM 등) — `base_url`·모델 이름 교체 |
| CSV 표 7개 | EMR 조회 API·데이터 웨어하우스. 도구 함수 안쪽만 교체 |
| `Context("P1")`: 연 환자에게만 묶인 도구 | 조회 권한·감사 로그와 함께 서버에서 보장 |
| 자동 점검 3종 | 배포 전 검증 + 운영 중 모니터링 지표 |

**토론**: 이 초안을 진료에 쓰려면 무엇이 더 필요할까요? (책임 소재, 의사 수정률 측정, 실패 사례 수집, 환자 동의 …)

⚠️ 무료 API에는 실제 환자 정보를 절대 넣지 마세요. 무료 등급의 입력은 서비스 개선에 쓰일 수 있습니다.